# YOLO detect notebook с background augmentation

Ноутбук расширяет выборку для детектора страниц: вырезает страницы по YOLO-seg разметке, кладет их на изображения столов, сохраняет новые изображения и аннотации, затем обучает YOLO.

In [ ]:
# Шаг 1: устанавливаем ultralytics в окружение Kaggle.
!pip install -q ultralytics


In [ ]:
# Шаг 1: импортируем библиотеки для работы с файлами, изображениями, разметкой и YOLO.
from pathlib import Path
import random
import shutil
import math
import json
import yaml
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from ultralytics import YOLO


In [ ]:
# Шаг 1: задаем пути Kaggle к объединенному датасету страниц, YOLO-разметке и фонам столов.
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".JPG", ".JPEG", ".PNG")

PAGE_DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/bobrzol123/hwr200_school_page_seg_yolo"),
    Path("/kaggle/input/datasets/bobrzol123/hwr200_school_page_seg_yolo/hwr200_school_page_seg_yolo"),
    Path("/kaggle/input/hwr200-school-page-seg-yolo/hwr200_school_page_seg_yolo"),
    Path("/kaggle/input/hwr200-school-page-seg-yolo"),
    Path("/kaggle/input/datasets"),
]


def has_page_dataset_layout(dataset_root: Path) -> bool:
    """
    Короткое описание:
        Проверяет, есть ли внутри dataset_root папки images и labels с файлами.
    Вход:
        dataset_root (Path): кандидат на корень датасета страниц.
    Выход:
        bool: True, если структура похожа на нужный YOLO-seg датасет.
    """
    images_dir = dataset_root / "images"
    labels_dir = dataset_root / "labels"
    if not images_dir.exists() or not labels_dir.exists():
        return False
    has_images = any(path.suffix in IMAGE_EXTENSIONS for path in images_dir.rglob("*"))
    has_labels = any(labels_dir.rglob("*.txt"))
    return has_images and has_labels


def resolve_page_dataset(candidates: list[Path]) -> Path:
    """
    Короткое описание:
        Находит корень объединенного датасета страниц после распаковки Kaggle.
    Вход:
        candidates (list[Path]): список возможных путей и родительских папок.
    Выход:
        Path: корень датасета, где лежат images и labels.
    """
    # Шаг 1: сначала проверяем точные ожидаемые пути.
    for candidate in candidates:
        if candidate.exists() and has_page_dataset_layout(candidate):
            return candidate

    # Шаг 2: если zip распакован с дополнительной вложенной папкой, ищем labels и берем ее parent.
    for candidate in candidates:
        if not candidate.exists():
            continue
        for labels_dir in candidate.rglob("labels"):
            dataset_root = labels_dir.parent
            if has_page_dataset_layout(dataset_root):
                return dataset_root

    checked = "\n".join(str(path) for path in candidates)
    raise FileNotFoundError(f"Не найден датасет hwr200_school_page_seg_yolo. Проверенные пути:\n{checked}")


PAGE_DATASET_SRC = resolve_page_dataset(PAGE_DATASET_CANDIDATES)
PAGE_IMG_SRC = PAGE_DATASET_SRC / "images"
PAGE_TXT_SRC = PAGE_DATASET_SRC / "labels"
PAGE_TXT_FALLBACK_SRC = PAGE_TXT_SRC
TABLE_IMG_SRC = Path("/kaggle/input/datasets/bobrzol123/dataset-background")
OUTPUT_DIR = Path("/kaggle/working")
TRAIN_LIST_PATH = OUTPUT_DIR / "train.txt"
VAL_LIST_PATH = OUTPUT_DIR / "val.txt"
TEST_LIST_PATH = OUTPUT_DIR / "test.txt"

# Шаг 2: задаем долю данных для быстрого прогона. Можно поставить 0.01, 0.10, 0.50 или 1.0.
USE_DATA_FRACTION = False  # True: берем только DATA_FRACTION исходных изображений для проверки пайплайна.
DATA_FRACTION = 0.01  # Доля исходных страниц и столов в быстром прогоне. 0.01 = 1%, 1.0 = все данные.
FAST_SYNTHETIC_COUNT = 32  # Сколько синтетических train-изображений делать при USE_DATA_FRACTION=True.
FULL_SYNTHETIC_COUNT = 2000  # Сколько синтетических train-изображений делать при USE_DATA_FRACTION=False.
AUGMENT_WORKERS = 1  # Число потоков для параллельной генерации синтетики на Kaggle.
MAX_PENDING_AUGMENT_TASKS = AUGMENT_WORKERS * 4  # Сколько готовящихся задач можно держать в памяти одновременно.

# Шаг 3: задаем разбиение оригинальных страниц. Сумма TRAIN_SIZE + VAL_SIZE + TEST_SIZE должна быть 1.0.
TRAIN_SIZE = 0.85  # Доля оригинальных изображений для обучения.
VAL_SIZE = 0.10  # Доля оригинальных изображений для validation.
TEST_SIZE = 0.05  # Доля оригинальных изображений для честного test.
RANDOM_SEED = 42  # Фиксирует случайность для повторяемого результата.

# Шаг 4: задаем гиперпараметры генерации синтетических изображений.
SYNTHETIC_COUNT = FAST_SYNTHETIC_COUNT if USE_DATA_FRACTION else FULL_SYNTHETIC_COUNT
DEBUG_SYNTHETIC_COUNT = 12  # Количество синтетических примеров для визуальной проверки.
ADD_TIGHT_PAGE_PATCHES_TO_TRAIN = False  # Не добавляем вырезанные страницы в train отдельными примерами.
MAX_TIGHT_PAGE_PATCHES = None  # None: добавить все page-patch из train; число: ограничить количество.
MAX_PAGES_ON_TABLE = 3  # Максимальное число страниц на одном фоне стола.
MAX_PLACE_TRIES = 20  # Число попыток поставить страницу без сильного пересечения.
MAX_BBOX_IOU = 0.08  # Максимальное пересечение bbox страниц на одном синтетическом изображении.
PAGE_SCALE_RANGE = (0.10, 1.05)  # Страница может быть маленькой или почти на весь стол.
PAGE_TARGET_AREA_FRACTION_RANGE = (0.03, 0.85)  # Доля площади стола, которую пытается занять страница.
PAGE_RANDOM_SCALE_PROBABILITY = 0.35  # Часть примеров масштабируется напрямую из PAGE_SCALE_RANGE.
PAGE_ROTATE_DEGREES = (-32, 32)  # Диапазон случайного поворота страницы в градусах.
MIN_PAGE_AREA = 5000  # Минимальная площадь вырезанной страницы в пикселях.

# Шаг 5: фильтры качества page-patch. Они защищают от ситуации, когда строка стала "страницей".
MIN_PAGE_ASPECT_RATIO = 0.25  # min(width / height) bbox страницы.
MAX_PAGE_ASPECT_RATIO = 4.0  # max(width / height) bbox страницы; тонкие полоски отбрасываем.
MIN_PAGE_IMAGE_AREA_FRACTION = 0.03  # bbox страницы должен занимать заметную часть исходного изображения.
MIN_MASK_FILL_RATIO = 0.25  # полигон должен заметно заполнять свой bbox, а не быть тонкой линией.
DEBUG_BAD_PATCH_COUNT = 20  # Сколько отвергнутых кандидатов сохранить для диагностики.
DEBUG_PAGE_POOL_COUNT = 20  # Сколько хороших patch сохранить для диагностики.

# Шаг 6: проверяем, как Kaggle распаковал датасет разметки страниц.
if not PAGE_TXT_SRC.exists() and PAGE_TXT_FALLBACK_SRC.exists():
    PAGE_TXT_SRC = PAGE_TXT_FALLBACK_SRC
print(f"Page labels: {PAGE_TXT_SRC}")
print(f"Use data fraction: {USE_DATA_FRACTION}")
print(f"Data fraction: {DATA_FRACTION}")
print(f"Split train/val/test: {TRAIN_SIZE:.2f}/{VAL_SIZE:.2f}/{TEST_SIZE:.2f}")

# Шаг 7: фиксируем генераторы случайных чисел.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Шаг 8: очищаем старые результаты в /kaggle/working, чтобы не смешивать разные запуски.
for stale_path in [
    OUTPUT_DIR / "images",
    OUTPUT_DIR / "labels",
    DEBUG_DIR if "DEBUG_DIR" in globals() else OUTPUT_DIR / "debug_synthetic_notebooks",
    PAGE_POOL_CACHE_DIR if "PAGE_POOL_CACHE_DIR" in globals() else OUTPUT_DIR / "page_pool_cache",
    TRAIN_LIST_PATH,
    VAL_LIST_PATH,
    TEST_LIST_PATH,
    OUTPUT_DIR / "data.yaml",
]:
    if stale_path.is_dir():
        shutil.rmtree(stale_path)
    elif stale_path.exists():
        stale_path.unlink()

# Шаг 9: создаем структуру YOLO-датасета в /kaggle/working только для синтетики.
for split in ["train", "val", "test"]:
    (OUTPUT_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# Шаг 10: создаем папки debug и кэша.
DEBUG_DIR = OUTPUT_DIR / "debug_synthetic_notebooks"
DEBUG_DIR.mkdir(parents=True, exist_ok=True)
BAD_PATCH_DEBUG_DIR = DEBUG_DIR / "bad_page_candidates"
GOOD_PATCH_DEBUG_DIR = DEBUG_DIR / "good_page_patches"
BAD_PATCH_DEBUG_DIR.mkdir(parents=True, exist_ok=True)
GOOD_PATCH_DEBUG_DIR.mkdir(parents=True, exist_ok=True)
PAGE_POOL_CACHE_DIR = OUTPUT_DIR / "page_pool_cache"
PAGE_POOL_CACHE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def list_images(directory: Path) -> list[Path]:
    """
    Короткое описание:
        Собирает все изображения из директории и ее подпапок.
    Вход:
        directory (Path): путь к папке с изображениями.
    Выход:
        list[Path]: отсортированный список путей к изображениям.
    """
    # Шаг 1: рекурсивно обходим папку и оставляем только известные расширения изображений.
    return sorted([p for p in directory.rglob("*") if p.suffix in IMAGE_EXTENSIONS])



def build_label_index(labels_dir: Path) -> dict[str, Path]:
    """
    Короткое описание:
        Строит индекс YOLO-label файлов по имени изображения без расширения.
    Вход:
        labels_dir (Path): директория с .txt файлами разметки.
    Выход:
        dict[str, Path]: словарь stem изображения -> путь к .txt разметке.
    """
    label_index = {}
    duplicates = 0

    # Шаг 1: ищем .txt рекурсивно, потому что Kaggle иногда кладет папку внутри папки.
    for label_path in sorted(labels_dir.rglob("*.txt")):
        if label_path.stem in label_index:
            duplicates += 1
            continue
        label_index[label_path.stem] = label_path

    print(f"Label files indexed: {len(label_index)}")
    if duplicates:
        print(f"Label duplicate stems skipped: {duplicates}")
    return label_index


def find_label_path(image_path: Path) -> Path | None:
    """
    Короткое описание:
        Находит .txt разметку для изображения сначала напрямую, потом через LABEL_INDEX.
    Вход:
        image_path (Path): путь к изображению.
    Выход:
        Path | None: путь к .txt файлу или None, если разметки нет.
    """
    # Шаг 1: быстрый путь для плоской структуры labels.
    direct_path = PAGE_TXT_SRC / f"{image_path.stem}.txt"
    if direct_path.exists():
        return direct_path

    # Шаг 2: fallback через рекурсивный индекс labels.
    return LABEL_INDEX.get(image_path.stem)

def read_yolo_segments(txt_path: Path, image_shape: tuple[int, int]) -> list[np.ndarray]:
    """
    Короткое описание:
        Читает YOLO-seg разметку и переводит нормированные координаты в пиксели.
    Вход:
        txt_path (Path): путь к .txt файлу YOLO-seg.
        image_shape (tuple[int, int]): размер изображения в формате (height, width).
    Выход:
        list[np.ndarray]: список полигонов в пиксельных координатах.
    """
    height, width = image_shape
    if not txt_path.exists():
        return []

    segments = []
    # Шаг 1: читаем строки YOLO-seg формата class x1 y1 x2 y2 ...
    for line in txt_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        values = [float(x) for x in parts[1:]]
        points = np.array(values, dtype=np.float32).reshape(-1, 2)
        # Шаг 2: переводим координаты из [0, 1] в пиксели исходного изображения.
        points[:, 0] *= width
        points[:, 1] *= height
        segments.append(points)
    return segments


def save_yolo_segments(txt_path: Path, segments: list[np.ndarray], image_shape: tuple[int, int]) -> None:
    """
    Короткое описание:
        Сохраняет полигоны страниц в YOLO-seg формате.
    Вход:
        txt_path (Path): путь, куда нужно записать .txt файл.
        segments (list[np.ndarray]): список полигонов в пиксельных координатах.
        image_shape (tuple[int, int]): размер изображения в формате (height, width).
    Выход:
        отсутствует.
    """
    height, width = image_shape
    lines = []
    # Шаг 1: ограничиваем координаты границами изображения и нормируем их.
    for polygon in segments:
        clipped = polygon.copy().astype(np.float32)
        clipped[:, 0] = np.clip(clipped[:, 0], 0, width - 1)
        clipped[:, 1] = np.clip(clipped[:, 1], 0, height - 1)
        normalized = clipped.copy()
        normalized[:, 0] /= width
        normalized[:, 1] /= height
        coords = " ".join(f"{v:.6f}" for v in normalized.reshape(-1))
        lines.append(f"0 {coords}")
    # Шаг 2: записываем одну строку на один объект класса page.
    txt_path.write_text("\n".join(lines), encoding="utf-8")


def polygon_bbox(points: np.ndarray) -> tuple[float, float, float, float]:
    """
    Короткое описание:
        Считает axis-aligned bbox для полигона.
    Вход:
        points (np.ndarray): массив точек полигона формы (N, 2).
    Выход:
        tuple[float, float, float, float]: bbox в формате (x1, y1, x2, y2).
    """
    # Шаг 1: берем минимальные и максимальные координаты по x и y.
    x1, y1 = points.min(axis=0)
    x2, y2 = points.max(axis=0)
    return float(x1), float(y1), float(x2), float(y2)


def polygon_area(points: np.ndarray) -> float:
    """
    Короткое описание:
        Считает площадь полигона формулой Гаусса.
    Вход:
        points (np.ndarray): массив точек полигона формы (N, 2).
    Выход:
        float: площадь полигона.
    """
    if len(points) < 3:
        return 0.0
    x = points[:, 0]
    y = points[:, 1]
    return float(abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))) * 0.5)


def validate_page_polygon(polygon: np.ndarray, image_shape: tuple[int, int]) -> tuple[bool, dict]:
    """
    Короткое описание:
        Проверяет, похож ли полигон на страницу, а не на строку или мусор.
    Вход:
        polygon (np.ndarray): полигон в пикселях.
        image_shape (tuple[int, int]): размер изображения в формате (height, width).
    Выход:
        tuple[bool, dict]: флаг валидности и диагностические признаки.
    """
    height, width = image_shape
    x1, y1, x2, y2 = polygon_bbox(polygon)
    bbox_w = max(0.0, x2 - x1)
    bbox_h = max(0.0, y2 - y1)
    bbox_area = bbox_w * bbox_h
    image_area = float(height * width)
    aspect_ratio = bbox_w / max(bbox_h, 1e-6)
    area_fraction = bbox_area / max(image_area, 1.0)
    poly_area = polygon_area(polygon)
    fill_ratio = poly_area / max(bbox_area, 1.0)

    reasons = []
    if bbox_area < MIN_PAGE_AREA:
        reasons.append("small_bbox_area")
    if not (MIN_PAGE_ASPECT_RATIO <= aspect_ratio <= MAX_PAGE_ASPECT_RATIO):
        reasons.append("bad_aspect_ratio")
    if area_fraction < MIN_PAGE_IMAGE_AREA_FRACTION:
        reasons.append("small_image_fraction")
    if fill_ratio < MIN_MASK_FILL_RATIO:
        reasons.append("thin_polygon_fill")

    return len(reasons) == 0, {
        "bbox_w": float(bbox_w),
        "bbox_h": float(bbox_h),
        "bbox_area": float(bbox_area),
        "aspect_ratio": float(aspect_ratio),
        "area_fraction": float(area_fraction),
        "polygon_area": float(poly_area),
        "fill_ratio": float(fill_ratio),
        "reasons": reasons,
    }


def save_candidate_debug(image: np.ndarray, polygon: np.ndarray, output_path: Path, color: tuple[int, int, int], text: str) -> None:
    """
    Короткое описание:
        Сохраняет debug-картинку кандидата page-patch на исходном изображении.
    Вход:
        image (np.ndarray): исходное изображение BGR.
        polygon (np.ndarray): полигон кандидата.
        output_path (Path): путь сохранения.
        color (tuple[int, int, int]): цвет контура BGR.
        text (str): подпись причины/статуса.
    Выход:
        отсутствует.
    """
    debug = image.copy()
    cv2.polylines(debug, [polygon.astype(np.int32)], isClosed=True, color=color, thickness=4)
    cv2.putText(debug, text[:120], (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 3, cv2.LINE_AA)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output_path), debug)


def bbox_iou(box_a: tuple[float, float, float, float], box_b: tuple[float, float, float, float]) -> float:
    """
    Короткое описание:
        Считает IoU двух прямоугольников.
    Вход:
        box_a (tuple[float, float, float, float]): первый bbox в формате (x1, y1, x2, y2).
        box_b (tuple[float, float, float, float]): второй bbox в формате (x1, y1, x2, y2).
    Выход:
        float: значение IoU от 0 до 1.
    """
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    # Шаг 1: считаем площадь пересечения.
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter = inter_w * inter_h
    # Шаг 2: считаем объединение и итоговый IoU.
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    return inter / max(area_a + area_b - inter, 1e-6)


In [ ]:
def extract_page_patch(image: np.ndarray, polygon: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray, dict] | None:
    """
    Короткое описание:
        Вырезает страницу по валидному полигону и строит локальную маску страницы.
    Вход:
        image (np.ndarray): исходное цветное изображение BGR.
        polygon (np.ndarray): полигон страницы в пиксельных координатах исходного изображения.
    Выход:
        tuple[np.ndarray, np.ndarray, np.ndarray, dict] | None: patch, mask, local_polygon, metrics или None.
    """
    # Шаг 1: проверяем, что полигон похож на страницу, а не на строку.
    is_valid, metrics = validate_page_polygon(polygon, image.shape[:2])
    if not is_valid:
        return None

    # Шаг 2: считаем ограничивающий прямоугольник страницы.
    x1, y1, x2, y2 = polygon_bbox(polygon)
    x1 = max(0, int(math.floor(x1)))
    y1 = max(0, int(math.floor(y1)))
    x2 = min(image.shape[1], int(math.ceil(x2)))
    y2 = min(image.shape[0], int(math.ceil(y2)))
    if x2 <= x1 or y2 <= y1:
        return None

    # Шаг 3: вырезаем patch и переносим полигон в локальную систему координат.
    patch = image[y1:y2, x1:x2].copy()
    local_polygon = polygon.copy()
    local_polygon[:, 0] -= x1
    local_polygon[:, 1] -= y1

    # Шаг 4: строим бинарную маску страницы внутри patch.
    mask = np.zeros(patch.shape[:2], dtype=np.uint8)
    cv2.fillPoly(mask, [local_polygon.astype(np.int32)], 255)
    metrics = dict(metrics)
    metrics["patch_shape"] = [int(patch.shape[0]), int(patch.shape[1])]
    return patch, mask, local_polygon, metrics


def rotate_patch_and_polygon(
    patch: np.ndarray,
    mask: np.ndarray,
    polygon: np.ndarray,
    angle: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Короткое описание:
        Поворачивает patch, маску и полигон на один и тот же угол.
    Вход:
        patch (np.ndarray): цветной фрагмент страницы.
        mask (np.ndarray): маска страницы для фрагмента.
        polygon (np.ndarray): полигон страницы в координатах patch.
        angle (float): угол поворота в градусах.
    Выход:
        tuple[np.ndarray, np.ndarray, np.ndarray]: повернутые patch, mask и polygon.
    """
    height, width = patch.shape[:2]
    center = (width / 2, height / 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    cos = abs(matrix[0, 0])
    sin = abs(matrix[0, 1])
    new_width = int(height * sin + width * cos)
    new_height = int(height * cos + width * sin)
    matrix[0, 2] += new_width / 2 - center[0]
    matrix[1, 2] += new_height / 2 - center[1]

    # Шаг 1: поворачиваем изображение с белой заливкой пустых областей.
    rotated_patch = cv2.warpAffine(
        patch,
        matrix,
        (new_width, new_height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=(255, 255, 255),
    )
    # Шаг 2: поворачиваем маску ближайшим соседом, чтобы не получить полутона.
    rotated_mask = cv2.warpAffine(
        mask,
        matrix,
        (new_width, new_height),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )

    # Шаг 3: применяем ту же affine-матрицу к точкам полигона.
    points_h = np.hstack([polygon.astype(np.float32), np.ones((len(polygon), 1), dtype=np.float32)])
    rotated_polygon = points_h @ matrix.T
    return rotated_patch, rotated_mask, rotated_polygon.astype(np.float32)


def resize_patch_and_polygon(
    patch: np.ndarray,
    mask: np.ndarray,
    polygon: np.ndarray,
    scale: float,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Короткое описание:
        Масштабирует patch, маску и полигон страницы.
    Вход:
        patch (np.ndarray): цветной фрагмент страницы.
        mask (np.ndarray): маска страницы для фрагмента.
        polygon (np.ndarray): полигон страницы в координатах patch.
        scale (float): коэффициент масштабирования.
    Выход:
        tuple[np.ndarray, np.ndarray, np.ndarray]: масштабированные patch, mask и polygon.
    """
    # Шаг 1: считаем новый размер и применяем resize к изображению и маске.
    new_width = max(8, int(round(patch.shape[1] * scale)))
    new_height = max(8, int(round(patch.shape[0] * scale)))
    resized_patch = cv2.resize(patch, (new_width, new_height), interpolation=cv2.INTER_AREA)
    resized_mask = cv2.resize(mask, (new_width, new_height), interpolation=cv2.INTER_NEAREST)
    return resized_patch, resized_mask, polygon * scale


def paste_patch(
    background: np.ndarray,
    patch: np.ndarray,
    mask: np.ndarray,
    x: int,
    y: int,
) -> None:
    """
    Короткое описание:
        Накладывает страницу на фон по маске.
    Вход:
        background (np.ndarray): изображение фона, изменяется на месте.
        patch (np.ndarray): изображение страницы.
        mask (np.ndarray): бинарная маска страницы.
        x (int): координата x левого верхнего угла вставки.
        y (int): координата y левого верхнего угла вставки.
    Выход:
        отсутствует.
    """
    # Шаг 1: берем область фона и смешиваем ее с patch только внутри маски страницы.
    roi = background[y:y + patch.shape[0], x:x + patch.shape[1]]
    alpha = (mask.astype(np.float32) / 255.0)[..., None]
    blended = patch.astype(np.float32) * alpha + roi.astype(np.float32) * (1.0 - alpha)
    background[y:y + patch.shape[0], x:x + patch.shape[1]] = blended.astype(np.uint8)


In [ ]:
def build_page_pool(page_images: list[Path]) -> tuple[list[dict], dict]:
    """
    Короткое описание:
        Собирает пул вырезанных страниц на диске и пишет подробный debug качества кандидатов.
    Вход:
        page_images (list[Path]): список путей к изображениям страниц.
    Выход:
        tuple[list[dict], dict]: page_pool и статистика сборки.
    """
    pool = []
    stats = {
        "images_total": len(page_images),
        "images_read_ok": 0,
        "labels_missing": 0,
        "polygons_total": 0,
        "polygons_kept": 0,
        "polygons_rejected": 0,
        "reject_reasons": {},
        "kept_examples": [],
        "rejected_examples": [],
    }

    # Шаг 1: читаем по одному изображению, вырезаем страницы и сразу пишем patch/mask на диск.
    for image_path in tqdm(page_images, desc="Build page pool"):
        image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
        if image is None:
            continue
        stats["images_read_ok"] += 1
        txt_path = find_label_path(image_path)
        if txt_path is None:
            stats["labels_missing"] += 1
            del image
            continue
        polygons = read_yolo_segments(txt_path, image.shape[:2])
        stats["polygons_total"] += len(polygons)

        # Шаг 2: каждый валидный patch сохраняем отдельно, чтобы не копить тяжелые массивы в памяти.
        for polygon_idx, polygon in enumerate(polygons):
            is_valid, metrics = validate_page_polygon(polygon, image.shape[:2])
            if not is_valid:
                stats["polygons_rejected"] += 1
                for reason in metrics["reasons"]:
                    stats["reject_reasons"][reason] = stats["reject_reasons"].get(reason, 0) + 1
                if len(stats["rejected_examples"]) < DEBUG_BAD_PATCH_COUNT:
                    debug_name = f"bad_{len(stats['rejected_examples']):03d}_{image_path.stem}_{polygon_idx}.jpg"
                    save_candidate_debug(
                        image,
                        polygon,
                        BAD_PATCH_DEBUG_DIR / debug_name,
                        (0, 0, 255),
                        ",".join(metrics["reasons"]),
                    )
                    stats["rejected_examples"].append({
                        "image_path": str(image_path),
                        "polygon_idx": int(polygon_idx),
                        "debug_path": str(BAD_PATCH_DEBUG_DIR / debug_name),
                        **metrics,
                    })
                continue

            item = extract_page_patch(image, polygon)
            if item is None:
                continue
            patch, mask, local_polygon, metrics = item
            item_id = f"{image_path.stem}_{polygon_idx:03d}_{len(pool):06d}"
            patch_path = PAGE_POOL_CACHE_DIR / f"{item_id}_patch.jpg"
            mask_path = PAGE_POOL_CACHE_DIR / f"{item_id}_mask.png"
            cv2.imwrite(str(patch_path), patch)
            cv2.imwrite(str(mask_path), mask)
            pool.append({
                "image_path": str(image_path),
                "patch_path": str(patch_path),
                "mask_path": str(mask_path),
                "polygon": local_polygon.astype(np.float32),
                "metrics": metrics,
            })
            stats["polygons_kept"] += 1

            # Шаг 3: сохраняем несколько хороших примеров, чтобы визуально убедиться, что это страницы.
            if len(stats["kept_examples"]) < DEBUG_PAGE_POOL_COUNT:
                good_name = f"good_{len(stats['kept_examples']):03d}_{image_path.stem}_{polygon_idx}.jpg"
                good_path = GOOD_PATCH_DEBUG_DIR / good_name
                debug_patch = patch.copy()
                cv2.polylines(debug_patch, [local_polygon.astype(np.int32)], True, (0, 255, 0), 4)
                cv2.imwrite(str(good_path), debug_patch)
                stats["kept_examples"].append({
                    "image_path": str(image_path),
                    "polygon_idx": int(polygon_idx),
                    "debug_path": str(good_path),
                    **metrics,
                })

            del patch, mask, local_polygon
        del image

    # Шаг 4: сохраняем JSON-отчет по page pool.
    stats_path = DEBUG_DIR / "page_pool_stats.json"
    stats_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Page pool stats saved: {stats_path}")
    print(json.dumps({k: stats[k] for k in ['images_total', 'images_read_ok', 'labels_missing', 'polygons_total', 'polygons_kept', 'polygons_rejected', 'reject_reasons']}, ensure_ascii=False, indent=2))
    return pool, stats


def load_page_pool_item(item: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray] | None:
    """
    Короткое описание:
        Загружает один patch страницы из дискового кэша.
    Вход:
        item (dict): элемент page_pool с путями patch_path, mask_path и polygon.
    Выход:
        tuple[np.ndarray, np.ndarray, np.ndarray] | None: patch, mask, polygon или None при ошибке чтения.
    """
    # Шаг 1: читаем только один выбранный patch и маску перед вставкой на фон.
    patch = cv2.imread(item["patch_path"], cv2.IMREAD_COLOR)
    mask = cv2.imread(item["mask_path"], cv2.IMREAD_GRAYSCALE)
    if patch is None or mask is None:
        return None
    polygon = item["polygon"].copy().astype(np.float32)
    return patch, mask, polygon



def choose_page_scale(patch: np.ndarray, background_shape: tuple[int, int, int]) -> tuple[float, float | None]:
    """
    Короткое описание:
        Выбирает масштаб страницы так, чтобы она могла быть и маленькой, и большой на фоне стола.
    Вход:
        patch (np.ndarray): вырезанная страница BGR.
        background_shape (tuple[int, int, int]): форма фонового изображения.
    Выход:
        tuple[float, float | None]: масштаб и целевая доля площади стола, если она использовалась.
    """
    # Шаг 1: иногда берем прямой случайный масштаб, чтобы сохранить разнообразие размеров.
    if random.random() < PAGE_RANDOM_SCALE_PROBABILITY:
        return random.uniform(*PAGE_SCALE_RANGE), None

    # Шаг 2: чаще выбираем желаемую долю площади стола и пересчитываем ее в scale.
    background_height, background_width = background_shape[:2]
    target_area_fraction = random.uniform(*PAGE_TARGET_AREA_FRACTION_RANGE)
    target_area = background_height * background_width * target_area_fraction
    patch_area = max(1.0, float(patch.shape[0] * patch.shape[1]))
    scale = math.sqrt(target_area / patch_area)
    scale = float(np.clip(scale, PAGE_SCALE_RANGE[0], PAGE_SCALE_RANGE[1]))
    return scale, target_area_fraction

def create_synthetic_image(
    background_path: Path,
    page_pool: list[dict],
) -> tuple[np.ndarray, list[np.ndarray], list[dict]] | None:
    """
    Короткое описание:
        Создает одно синтетическое изображение: кладет одну или несколько страниц на стол.
    Вход:
        background_path (Path): путь к изображению стола.
        page_pool (list[dict]): пул метаданных вырезанных страниц на диске.
    Выход:
        tuple[np.ndarray, list[np.ndarray], list[dict]] | None: изображение, полигоны, debug-метаданные или None.
    """
    background = cv2.imread(str(background_path), cv2.IMREAD_COLOR)
    if background is None or not page_pool:
        return None

    result = background.copy()
    height, width = result.shape[:2]
    placed_polygons = []
    placed_boxes = []
    placed_debug = []
    page_count = random.randint(1, MAX_PAGES_ON_TABLE)

    # Шаг 1: пытаемся поставить несколько страниц без сильного пересечения.
    for _ in range(page_count):
        for _ in range(MAX_PLACE_TRIES):
            item = random.choice(page_pool)
            loaded_item = load_page_pool_item(item)
            if loaded_item is None:
                continue
            patch_src, mask_src, polygon_src = loaded_item
            scale, target_area_fraction = choose_page_scale(patch_src, result.shape)
            angle = random.uniform(*PAGE_ROTATE_DEGREES)

            # Шаг 2: случайно масштабируем и поворачиваем только текущую страницу.
            patch, mask, polygon = resize_patch_and_polygon(patch_src, mask_src, polygon_src, scale)
            patch, mask, polygon = rotate_patch_and_polygon(patch, mask, polygon, angle)
            del patch_src, mask_src, polygon_src

            if patch.shape[0] >= height or patch.shape[1] >= width:
                del patch, mask, polygon
                continue

            # Шаг 3: выбираем позицию на столе и проверяем пересечения с уже поставленными страницами.
            x = random.randint(0, width - patch.shape[1])
            y = random.randint(0, height - patch.shape[0])
            final_polygon = polygon.copy()
            final_polygon[:, 0] += x
            final_polygon[:, 1] += y
            final_box = polygon_bbox(final_polygon)

            if any(bbox_iou(final_box, old_box) > MAX_BBOX_IOU for old_box in placed_boxes):
                del patch, mask, polygon, final_polygon
                continue

            # Шаг 4: вставляем страницу и сохраняем ее новый полигон.
            paste_patch(result, patch, mask, x, y)
            placed_polygons.append(final_polygon)
            placed_boxes.append(final_box)
            placed_debug.append({
                "source_image": item["image_path"],
                "patch_path": item["patch_path"],
                "x": int(x),
                "y": int(y),
                "scale": float(scale),
                "target_area_fraction": None if target_area_fraction is None else float(target_area_fraction),
                "angle": float(angle),
                "patch_shape": [int(patch.shape[0]), int(patch.shape[1])],
                "background_shape": [int(height), int(width)],
                "source_metrics": item.get("metrics", {}),
            })
            del patch, mask, polygon
            break

    if not placed_polygons:
        return None
    return result, placed_polygons, placed_debug


def draw_debug(image: np.ndarray, polygons: list[np.ndarray], output_path: Path, metadata: list[dict] | None = None) -> None:
    """
    Короткое описание:
        Сохраняет debug-картинку с зелеными полигонами страниц и JSON-метаданными.
    Вход:
        image (np.ndarray): изображение для визуализации.
        polygons (list[np.ndarray]): список полигонов страниц.
        output_path (Path): путь для сохранения debug-изображения.
        metadata (list[dict] | None): сведения об источниках страниц.
    Выход:
        отсутствует.
    """
    # Шаг 1: рисуем контуры страниц и нумерацию.
    debug = image.copy()
    for idx, polygon in enumerate(polygons):
        pts = polygon.astype(np.int32)
        cv2.polylines(debug, [pts], isClosed=True, color=(0, 255, 0), thickness=4)
        x1, y1, _, _ = polygon_bbox(polygon)
        cv2.putText(debug, str(idx), (int(x1), max(30, int(y1))), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    cv2.imwrite(str(output_path), debug)

    # Шаг 2: рядом сохраняем JSON с источниками patch, scale, angle и фильтрами качества.
    if metadata is not None:
        output_path.with_suffix(".json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")


In [ ]:
# Шаг 1: собираем изображения страниц и столов.
page_images = list_images(PAGE_IMG_SRC)
table_images = list_images(TABLE_IMG_SRC)
LABEL_INDEX = build_label_index(PAGE_TXT_SRC)
print(f"Page images before label filter: {len(page_images)}")
print(f"Table images before fraction filter: {len(table_images)}")

# Шаг 2: оставляем только изображения, для которых реально найдена YOLO-разметка страницы.
labeled_page_images = []
missing_label_examples = []
for image_path in page_images:
    if find_label_path(image_path) is None:
        if len(missing_label_examples) < 10:
            missing_label_examples.append(str(image_path))
        continue
    labeled_page_images.append(image_path)
page_images = labeled_page_images
print(f"Page images with labels: {len(page_images)}")
if missing_label_examples:
    print("First images without labels:")
    for item in missing_label_examples:
        print(f"  {item}")

# Шаг 3: проверяем доли разбиения, чтобы train/val/test были управляемыми и честными.
split_sum = TRAIN_SIZE + VAL_SIZE + TEST_SIZE
if abs(split_sum - 1.0) > 1e-6:
    raise ValueError(f"TRAIN_SIZE + VAL_SIZE + TEST_SIZE должен быть 1.0, сейчас {split_sum}")
if not (0.0 < DATA_FRACTION <= 1.0):
    raise ValueError("DATA_FRACTION должен быть в диапазоне (0, 1].")

# Шаг 4: при необходимости берем любую заданную долю размеченных данных, а не только жесткий 1%.
if USE_DATA_FRACTION and DATA_FRACTION < 1.0:
    rng = random.Random(RANDOM_SEED)
    page_sample_count = max(3, int(round(len(page_images) * DATA_FRACTION)))
    table_sample_count = max(1, int(round(len(table_images) * DATA_FRACTION)))
    page_images = sorted(rng.sample(page_images, min(page_sample_count, len(page_images))))
    table_images = sorted(rng.sample(table_images, min(table_sample_count, len(table_images))))
    print(f"[DATA FRACTION MODE] fraction={DATA_FRACTION}, synthetic_count={SYNTHETIC_COUNT}")

print(f"Page images used: {len(page_images)}")
print(f"Table images used: {len(table_images)}")
print(f"Synthetic count: {SYNTHETIC_COUNT}")

# Шаг 5: проверяем, что после фильтров осталось достаточно данных.
if len(page_images) < 3:
    raise RuntimeError(
        "После поиска labels осталось меньше 3 изображений. "
        f"PAGE_TXT_SRC={PAGE_TXT_SRC}, label_files={len(LABEL_INDEX)}, labeled_images={len(page_images)}"
    )
if not table_images:
    raise RuntimeError(f"Не найдены изображения столов в TABLE_IMG_SRC={TABLE_IMG_SRC}")

# Шаг 6: делим оригинальные страницы на train, val и test.
train_val_imgs, test_imgs = train_test_split(
    page_images,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
)
relative_val_size = VAL_SIZE / max(TRAIN_SIZE + VAL_SIZE, 1e-6)
train_imgs, val_imgs = train_test_split(
    train_val_imgs,
    test_size=relative_val_size,
    random_state=RANDOM_SEED,
)

print(f"Train original images: {len(train_imgs)}")
print(f"Val original images: {len(val_imgs)}")
print(f"Test original images: {len(test_imgs)}")

def write_image_list(list_path: Path, image_paths: list[Path]) -> None:
    """
    Короткое описание:
        Сохраняет txt со списком изображений для YOLO без копирования исходных файлов.
    Вход:
        list_path (Path): путь к train.txt/val.txt/test.txt.
        image_paths (list[Path]): список абсолютных путей изображений.
    Выход:
        отсутствует.
    """
    list_path.write_text("\n".join(str(path) for path in image_paths) + "\n", encoding="utf-8")


# Шаг 7: сохраняем списки исходных изображений. Сами картинки и labels остаются в /kaggle/input.
write_image_list(TRAIN_LIST_PATH, train_imgs)
write_image_list(VAL_LIST_PATH, val_imgs)
write_image_list(TEST_LIST_PATH, test_imgs)
print(f"Train list: {TRAIN_LIST_PATH} ({len(train_imgs)} original images)")
print(f"Val list: {VAL_LIST_PATH} ({len(val_imgs)} original images)")
print(f"Test list: {TEST_LIST_PATH} ({len(test_imgs)} original images)")

# Шаг 8: строим пул вырезанных страниц только из train, чтобы не подмешивать val/test в синтетику.
# В RAM остается только список путей и полигонов, сами patch/mask лежат на диске.
page_pool, page_pool_stats = build_page_pool(train_imgs)
print(f"Page patches cached on disk: {len(page_pool)}")
print(f"Page pool cache: {PAGE_POOL_CACHE_DIR}")

if len(page_pool) == 0:
    stats_message = json.dumps(page_pool_stats, ensure_ascii=False, indent=2)[:4000]
    raise RuntimeError(
        "Page pool пустой. Ни один полигон не прошел фильтры качества.\n"
        f"PAGE_TXT_SRC={PAGE_TXT_SRC}\n"
        f"DEBUG_BAD_PATCH_DIR={BAD_PATCH_DEBUG_DIR}\n"
        f"page_pool_stats(first 4000 chars)=\n{stats_message}"
    )


In [ ]:
# Шаг 1: вырезанные страницы используются только как pool для синтетики и не добавляются отдельными train-примерами.
tight_patch_saved_count = 0
print(f"Tight page patches saved to train: {tight_patch_saved_count}")


In [ ]:
def create_synthetic_task(idx: int) -> dict | None:
    """
    Короткое описание:
        Создает одно синтетическое изображение для параллельной генерации.
    Вход:
        idx (int): номер синтетического изображения.
    Выход:
        dict | None: данные изображения, разметки и debug или None при неудаче.
    """
    # Шаг 1: выбираем случайный фон и собираем синтетическое изображение.
    background_path = random.choice(table_images)
    item = create_synthetic_image(background_path, page_pool)
    if item is None:
        return None

    synthetic_image, synthetic_polygons, placed_debug = item
    image_name = f"synthetic_table_{idx:06d}.jpg"
    label_name = f"synthetic_table_{idx:06d}.txt"
    return {
        "idx": int(idx),
        "image_name": image_name,
        "label_name": label_name,
        "background_path": str(background_path),
        "synthetic_image": synthetic_image,
        "synthetic_polygons": synthetic_polygons,
        "placed_debug": placed_debug,
    }


def save_synthetic_task_result(task_result: dict, debug_idx: int | None) -> dict:
    """
    Короткое описание:
        Сохраняет результат синтетической аугментации в YOLO train-датасет.
    Вход:
        task_result (dict): результат create_synthetic_task.
        debug_idx (int | None): номер debug-картинки или None.
    Выход:
        dict: короткая metadata-запись для synthetic_generation_report.json.
    """
    synthetic_image = task_result["synthetic_image"]
    synthetic_polygons = task_result["synthetic_polygons"]
    placed_debug = task_result["placed_debug"]
    image_name = task_result["image_name"]
    label_name = task_result["label_name"]

    # Шаг 1: сохраняем картинку и YOLO-seg label с уникальными именами.
    cv2.imwrite(str(OUTPUT_DIR / "images" / "train" / image_name), synthetic_image)
    save_yolo_segments(
        OUTPUT_DIR / "labels" / "train" / label_name,
        synthetic_polygons,
        synthetic_image.shape[:2],
    )

    # Шаг 2: первые удачные примеры сохраняем отдельно для визуальной проверки.
    if debug_idx is not None:
        draw_debug(
            synthetic_image,
            synthetic_polygons,
            DEBUG_DIR / f"debug_{debug_idx:03d}.jpg",
            metadata=placed_debug,
        )

    return {
        "image_path": str(OUTPUT_DIR / "images" / "train" / image_name),
        "label_path": str(OUTPUT_DIR / "labels" / "train" / label_name),
        "image_name": image_name,
        "label_name": label_name,
        "background_path": task_result["background_path"],
        "objects": placed_debug,
    }


# Шаг 1: генерируем синтетические изображения только в train параллельно и с ограничением очереди.
debug_saved_count = 0
synthetic_saved_count = 0
synthetic_debug_metadata = []
next_task_idx = 0
pending_tasks = set()
max_pending_tasks = max(1, int(MAX_PENDING_AUGMENT_TASKS))
augment_workers = max(1, int(AUGMENT_WORKERS))

print(f"Parallel synthetic generation: workers={augment_workers}, max_pending={max_pending_tasks}")
with ThreadPoolExecutor(max_workers=augment_workers) as executor:
    with tqdm(total=SYNTHETIC_COUNT, desc="Create synthetic") as progress_bar:
        # Шаг 2: держим в памяти не больше max_pending_tasks больших изображений.
        while next_task_idx < SYNTHETIC_COUNT or pending_tasks:
            while next_task_idx < SYNTHETIC_COUNT and len(pending_tasks) < max_pending_tasks:
                pending_tasks.add(executor.submit(create_synthetic_task, next_task_idx))
                next_task_idx += 1

            done_tasks, pending_tasks = wait(pending_tasks, return_when=FIRST_COMPLETED)
            for future in done_tasks:
                task_result = future.result()
                progress_bar.update(1)
                if task_result is None:
                    continue

                debug_idx = debug_saved_count if debug_saved_count < DEBUG_SYNTHETIC_COUNT else None
                metadata_item = save_synthetic_task_result(task_result, debug_idx)
                synthetic_debug_metadata.append(metadata_item)
                synthetic_saved_count += 1
                if debug_idx is not None:
                    debug_saved_count += 1

                # Шаг 3: явно отпускаем тяжелые массивы после записи на диск.
                del task_result

# Шаг 4: обновляем train.txt: оригинальные train-изображения из /kaggle/input + синтетика из /kaggle/working.
synthetic_image_paths = [Path(item["image_path"]) for item in synthetic_debug_metadata]
write_image_list(TRAIN_LIST_PATH, list(train_imgs) + synthetic_image_paths)

# Шаг 5: сохраняем общий JSON по синтетике.
synthetic_report_path = DEBUG_DIR / "synthetic_generation_report.json"
synthetic_report_path.write_text(json.dumps(synthetic_debug_metadata, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Synthetic saved: {synthetic_saved_count}")
print(f"Debug saved: {debug_saved_count}")
print(f"Debug saved to: {DEBUG_DIR}")
print(f"Synthetic report: {synthetic_report_path}")


In [ ]:
# Шаг 1: показываем debug-примеры синтетики прямо в ноутбуке.
debug_images = sorted(DEBUG_DIR.glob("debug_*.jpg"))[:DEBUG_SYNTHETIC_COUNT]
print(f"Debug images found: {len(debug_images)}")
print(f"Good page patch debug: {GOOD_PATCH_DEBUG_DIR}")
print(f"Bad page candidate debug: {BAD_PATCH_DEBUG_DIR}")
print(f"Page pool stats: {DEBUG_DIR / 'page_pool_stats.json'}")
print(f"Synthetic report: {DEBUG_DIR / 'synthetic_generation_report.json'}")

if not debug_images:
    raise RuntimeError("Debug-изображения не найдены. Проверь page_pool, table_images и SYNTHETIC_COUNT.")

cols = min(2, len(debug_images))
rows = max(1, math.ceil(len(debug_images) / cols))
plt.figure(figsize=(14, 7 * rows))
for i, image_path in enumerate(debug_images):
    image_bgr = cv2.imread(str(image_path))
    if image_bgr is None:
        continue
    image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    plt.subplot(rows, cols, i + 1)
    plt.imshow(image)
    plt.title(image_path.name)
    plt.axis("off")
plt.tight_layout()
plt.show()

# Шаг 2: показываем несколько хороших page-patch, из которых строится синтетика.
good_patch_images = sorted(GOOD_PATCH_DEBUG_DIR.glob("*.jpg"))[:min(8, DEBUG_PAGE_POOL_COUNT)]
if good_patch_images:
    cols = min(4, len(good_patch_images))
    rows = max(1, math.ceil(len(good_patch_images) / cols))
    plt.figure(figsize=(16, 4 * rows))
    for i, image_path in enumerate(good_patch_images):
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            continue
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(image_path.name)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Шаг 3: показываем реальные YOLO-label файлы синтетики, чтобы проверить, что обучение увидит то же самое.
label_debug_images = sorted((OUTPUT_DIR / "images" / "train").glob("synthetic_table_*.jpg"))[:min(6, DEBUG_SYNTHETIC_COUNT)]
if label_debug_images:
    cols = min(2, len(label_debug_images))
    rows = max(1, math.ceil(len(label_debug_images) / cols))
    plt.figure(figsize=(14, 7 * rows))
    for i, image_path in enumerate(label_debug_images):
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            continue
        label_path = OUTPUT_DIR / "labels" / "train" / f"{image_path.stem}.txt"
        polygons = read_yolo_segments(label_path, image_bgr.shape[:2])
        for polygon in polygons:
            cv2.polylines(image_bgr, [polygon.astype(np.int32)], True, (0, 255, 0), 5)
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        plt.subplot(rows, cols, i + 1)
        plt.imshow(image)
        plt.title(f"YOLO label: {image_path.name}")
        plt.axis("off")
    plt.tight_layout()
    plt.show()


In [ ]:
# Шаг 1: создаем data.yaml для обучения YOLO.
data = {
    "path": str(OUTPUT_DIR),
    "train": str(TRAIN_LIST_PATH),
    "val": str(VAL_LIST_PATH),
    "test": str(TEST_LIST_PATH),
    "nc": 1,
    "names": ["page"],
}

# Шаг 2: сохраняем YAML в /kaggle/working.
file_path = OUTPUT_DIR / "data.yaml"
with open(file_path, "w", encoding="utf-8") as file:
    yaml.dump(data, file, default_flow_style=False, allow_unicode=True, sort_keys=False)

print(file_path.read_text())


In [ ]:
# Шаг 1: проверяем итоговое количество изображений после расширения датасета.
def count_list_items(list_path: Path) -> int:
    """
    Короткое описание:
        Считает непустые строки в txt-списке YOLO.
    Вход:
        list_path (Path): путь к train.txt/val.txt/test.txt.
    Выход:
        int: количество путей в списке.
    """
    return len([line for line in list_path.read_text(encoding="utf-8").splitlines() if line.strip()])


train_count = count_list_items(TRAIN_LIST_PATH)
val_count = count_list_items(VAL_LIST_PATH)
test_count = count_list_items(TEST_LIST_PATH)
synthetic_train_count = len(list((OUTPUT_DIR / "images" / "train").glob("synthetic_table_*.jpg")))
print(f"Train images in train.txt: {train_count}")
print(f"Val images in val.txt: {val_count}")
print(f"Test images in test.txt: {test_count}")
print(f"Synthetic train images: {synthetic_train_count}")
print(f"Original train images are not copied to /kaggle/working: {len(train_imgs)}")


In [ ]:
!rm -rf /kaggle/working/augmented_yolo_dataset_package
!rm -rf /kaggle/working/debug_synthetic_notebooks
!rm -rf /kaggle/working/page_pool_cache
!rm -rf /kaggle/working/my_experiment
!rm -f /kaggle/working/debug_prediction.jpg
!du -sh /kaggle/working/*


In [ ]:
# Шаг 1: загружаем базовую YOLOv8 segmentation модель.
model = YOLO("yolov8n-seg.pt")


In [ ]:
# Шаг 1: запускаем обучение YOLO на оригинальных и синтетических изображениях.
!yolo train model=yolov8n-seg.pt data=/kaggle/working/data.yaml epochs=5 imgsz=640 device=0,1 batch=4 project=/kaggle/working/my_experiment \
    augment=True \
    hsv_h=0.02 hsv_s=0.6 hsv_v=0.4 \
    degrees=35 \
    translate=0.1 \
    scale=0.45 \
    shear=4 \
    perspective=0.0005 \
    flipud=0.0 \
    fliplr=0.3 \
    mosaic=0.5 \
    mixup=0.05


In [ ]:
# Шаг 1: загружаем лучший вес после обучения и проверяем результат на одном val-изображении.
model_path = "/kaggle/working/my_experiment/train/weights/best.pt"
model = YOLO(model_path)

# Шаг 2: сохраняем и показываем предсказание.
sample_image = sorted((OUTPUT_DIR / "images" / "val").glob("*"))[0]
results = model(str(sample_image), imgsz=1024)
for result in results:
    result.save(filename="/kaggle/working/debug_prediction.jpg")
    result.show()


In [ ]:
from pathlib import Path
import shutil

WORK = Path("/kaggle/working")

# Оставляем только это.
KEEP = {
    "images",
    "labels",
    "data.yaml",
    "train.txt",
    "val.txt",
    "test.txt",
    "manifest.json",
}

for path in WORK.iterdir():
    if path.name in KEEP:
        continue

    if path.is_dir():
        shutil.rmtree(path)
        print(f"removed dir: {path}")
    else:
        path.unlink()
        print(f"removed file: {path}")

print("\nAfter cleanup:")
!du -sh /kaggle/working/*
